# Producción

En esta etapa se preparará el modelo seleccionado para su utilización en un entorno productivo.

Para ello se guardará el modelo entrenado, se construirá un pipeline de inferencia y se desarrollará una API utilizando FastAPI.

Posteriormente, la aplicación será containerizada mediante Docker y desplegada en la nube utilizando AWS.

El objetivo final es disponer de un servicio capaz de recibir nuevas transacciones y predecir automáticamente si presentan riesgo de fraude.

## Guardado del Modelo

Una vez seleccionado Random Forest como modelo final, se procede a almacenarlo en disco para permitir su reutilización sin necesidad de volver a entrenarlo.

Esto facilitará su integración posterior dentro del pipeline de producción y de la API de predicción.

In [2]:
import joblib

In [3]:
%run 04_modeling.ipynb

(283726, 30)
(283726,)
X_train: (226980, 30)
X_test: (56746, 30)
y_train: (226980,)
y_test: (56746,)
Original
Class
0    0.998333
1    0.001667
Name: proportion, dtype: float64

Train
Class
0    0.998335
1    0.001665
Name: proportion, dtype: float64

Test
Class
0    0.998326
1    0.001674
Name: proportion, dtype: float64
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.85      0.58      0.69        95

    accuracy                           1.00     56746
   macro avg       0.92      0.79      0.84     56746
weighted avg       1.00      1.00      1.00     56746

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.97      0.73      0.83        95

    accuracy                           1.00     56746
   macro avg       0.99      0.86      0.92     56746
weighted avg       1.00      1.00      1.00     56746

Note: you may need to 

In [4]:
joblib.dump(modelo_rf, "modelo_fraude.pkl")

['modelo_fraude.pkl']

In [5]:
import os

os.listdir()

['01_eda.ipynb',
 '02_cleaning.ipynb',
 '03_feature_engineering.ipynb',
 '04_modeling.ipynb',
 '05_evaluation.ipynb',
 '06_production.ipynb',
 'modelo_fraude.pkl']

## Carga del Modelo

Una vez almacenado el modelo entrenado, es importante verificar que puede cargarse correctamente desde disco.

Esto permite reutilizar el modelo en aplicaciones de producción sin necesidad de repetir el proceso de entrenamiento.

In [6]:
modelo_cargado = joblib.load("modelo_fraude.pkl")

## Validación del Modelo Cargado

Una vez cargado el modelo desde disco, se verifica que continúe funcionando correctamente realizando predicciones sobre el conjunto de prueba.

Esta validación garantiza que el proceso de serialización y carga del modelo no afectó su comportamiento.

In [7]:
predicciones = modelo_cargado.predict(X_test)

predicciones[:10]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

## Pipeline

En un entorno productivo, las transformaciones aplicadas a los datos y el modelo de Machine Learning deben ejecutarse siempre en el mismo orden.

Para garantizar la consistencia del proceso de inferencia, se construirá un Pipeline que integre las etapas necesarias antes de generar una predicción.

In [8]:
from sklearn.pipeline import Pipeline

In [9]:
pipeline = Pipeline([
    ("modelo", modelo_cargado)
])

In [10]:
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If

## FastAPI

Para exponer el modelo como un servicio accesible desde aplicaciones externas, se utilizará FastAPI.

La API permitirá recibir nuevas transacciones mediante peticiones HTTP y devolver una predicción indicando si existe riesgo de fraude.

In [11]:
%pip install fastapi uvicorn

Note: you may need to restart the kernel to use updated packages.


In [20]:
import fastapi

## Creación de la API

Se crea una API utilizando FastAPI para exponer el modelo de detección de fraude.

La API permitirá recibir información de una transacción y devolver una predicción indicando si la operación presenta riesgo de fraude.

In [ ]:
from fastapi import FastAPI

app = FastAPI()

from pydantic import BaseModel

class Transaction(BaseModel):
    Time: float
    V1: float
    V2: float
    V3: float
    V4: float
    V5: float
    V6: float
    V7: float
    V8: float
    V9: float
    V10: float
    V11: float
    V12: float
    V13: float
    V14: float
    V15: float
    V16: float
    V17: float
    V18: float
    V19: float
    V20: float
    V21: float
    V22: float
    V23: float
    V24: float
    V25: float
    V26: float
    V27: float
    V28: float
    Amount_scaled: float

In [12]:
@app.get("/")
def home():
    return {"mensaje": "API de detección de fraude funcionando"}

In [14]:
@app.post("/predict")
def predict(transaction: Transaction):
    datos = pd.DataFrame([transaction.model_dump()])
    prediccion = modelo_cargado.predict(datos)
    return {"resultado": int(prediccion[0])}

In [15]:
X_test.columns

Index(['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',
       'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20',
       'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28',
       'Amount_scaled'],
      dtype='str')

In [16]:
import pandas as pd

In [17]:
ejemplo = X_test.iloc[[0]]

ejemplo

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount_scaled
86568,61290.0,1.228821,-0.063408,0.274145,0.647465,-0.048135,0.372073,-0.224231,0.079939,0.640759,...,-0.096566,-0.129554,-0.083779,-0.151661,-0.700372,0.59855,0.491409,0.002989,0.001782,-0.146016


In [18]:
modelo_cargado.predict(ejemplo)

array([0])

In [19]:
import json

ejemplo = X_test.iloc[0].to_dict()

with open("../sample_request.json", "w") as f:
    json.dump(ejemplo, f, indent=4)

In [23]:
X_test[y_test == 1].head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount_scaled
116139,74159.0,-1.548788,1.808698,-0.953509,2.213085,-2.015728,-0.913457,-2.356013,1.197169,-1.678374,...,0.390786,0.855138,0.774745,0.059037,0.343200,-0.468938,-0.278338,0.625922,0.395573,0.764011
95597,65385.0,-2.923827,1.524837,-3.018758,3.289291,-5.755542,2.218276,-0.509995,-3.569444,-1.016592,...,-0.447039,-0.511657,-0.122724,-4.288639,0.563797,-0.949451,-0.204532,1.510206,-0.324706,18.526631
240222,150494.0,1.852889,1.069593,-1.776101,4.617410,0.770413,-0.400859,-0.040970,0.089510,-0.217705,...,-0.288392,-0.157869,-0.176244,0.027437,-0.468006,0.058063,0.148263,0.042278,0.040573,-0.292032
203700,134928.0,1.204934,3.238070,-6.010324,5.720847,1.548400,-2.321064,-0.781880,0.076619,-2.976249,...,0.338161,0.098341,-0.845866,-0.031228,0.421146,0.388361,0.056035,0.491828,0.340847,-0.305938
249239,154309.0,-0.082983,-3.935919,-2.616709,0.163310,-1.400952,-0.809419,1.501580,-0.471000,1.519743,...,1.878612,0.702672,-0.182305,-0.921017,0.111635,-0.071622,-1.125881,-0.170947,0.126221,14.949103


In [24]:
import json

ejemplo_fraude = X_test[y_test == 1].iloc[0].to_dict()

with open("../sample_request_fraud.json", "w") as f:
    json.dump(ejemplo_fraude, f, indent=4)